# 16_Task8_Step4 — Làm mượt kết quả khi chạy webcam

**Mục tiêu:** nhãn hiển thị trên webcam bớt nhảy qua lại (ví dụ rock → scissors → rock) khi tay chưa đổi, mà không làm mất lợi thế đoán sớm.

- Model nền: `task8_step2_distance` (`Data_321/checkpoints_v2/task8_step2_distance/TCN.pt`). **Không train, không tạo checkpoint mới, không ghi file nào vào `Data_321`.**
- Hướng làm (chỉ xử lý phía sau model, làm trên xác suất 3 lớp):
  1. **EMA:** trộn xác suất khung mới với xác suất đã làm mượt trước đó (`alpha` càng nhỏ càng mượt, càng trễ).
  2. **Giữ nhãn (hysteresis):** chỉ đổi nhãn đang hiển thị khi lớp mới đạt ≥ 0.5, hơn lớp hiện tại ít nhất `margin`, và kéo dài `hold` khung webcam liên tiếp.
  3. **Reset khi mất tay:** xóa trạng thái, lần ra tay sau bắt đầu lại từ đầu.
- Replay ở **30 fps** để mô phỏng webcam: lấy mỗi khung thứ 4 của landmark 120 fps, nội suy lại lên 120 fps như webcam đang làm, rồi đọc kết quả tại các khung webcam thật.
- Chọn tham số trên tập **val**, báo cáo trên tập **test**, so với chính model nền không làm mượt.
- Giao diện webcam giống `08_WebcamWide`, ngưỡng hiển thị màu **0.6**, phím `s` bật/tắt làm mượt, có bộ đếm số lần đổi nhãn (raw / mượt) để so bằng mắt.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 11. Cell 11 mở cửa sổ webcam.

| Cell | Việc |
|---|---|
| 1–3 | Chuẩn bị, nạp model, kiểm tra acc_all ≈ 93.2% |
| 4 | Hàm làm mượt + tự kiểm tra bằng dữ liệu giả |
| 5 | Replay val/test ở 30 fps |
| 6 | Hàm chấm điểm + điểm của model nền (không làm mượt) |
| 7 | Thử nhiều bộ tham số trên val, chọn bộ tốt nhất |
| 8 | Báo cáo trên test, kết luận sơ bộ |
| 9–11 | Webcam |

In [4]:
#cell 1: import, bật CUDA, kiểm tra các file cần có (notebook này không train, không tạo checkpoint)
import math, time, collections
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
TASK = BASE / "hand_landmarker.task"
CKPT = BASE / "checkpoints_v2" / "task8_step2_distance" / "TCN.pt"
for p in (BASE / "manifest.csv", TASK, CKPT):
    assert p.exists(), f"Không thấy: {p}"
m = pd.read_csv(BASE / "manifest.csv")
FPS_DATA, FPS_CAM = 120, 30
print("OK | GPU:", torch.cuda.get_device_name(0), "| video:", len(m))

OK | GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | video: 215


In [5]:
#cell 2: đặc trưng 546 (giống 13_Task8_Step2), định nghĩa TCN, nạp task8_step2_distance
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)

def features_from_pos(pos, hand_mask):
    valid = hand_mask[:, :, None, None]
    pos = torch.where(valid, pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]
    pos_flat = pos.reshape(b, T, 63)
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(pos_flat)
    vel[:, 1:] = (pos_flat[:, 1:] - pos_flat[:, :-1]) * pair_valid
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I] - pos[:, :, PAIR_J], dim=-1) * hand_mask[:, :, None]
    dist_vel = torch.zeros_like(dist)
    dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid
    return torch.cat([pos_flat, vel, dist, dist_vel], dim=-1)

def feats_from_raw(lm):
    """lm: (B,T,21,3) tọa độ gốc, NaN = mất tay -> (đặc trưng (B,T,546), mask (B,T))."""
    has = ~torch.isnan(lm[:, :, 0, 0])
    q = lm - lm[:, :, :1]
    q = q / torch.linalg.vector_norm(q[:, :, 9], dim=-1)[:, :, None, None].clamp(min=1e-8)
    q = torch.nan_to_num(q)
    return features_from_pos(q, has), has

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=546, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

CLASSES = ["paper", "rock", "scissors"]
ck = torch.load(CKPT, map_location=dev, weights_only=False)
net = TCN().to(dev); net.load_state_dict(ck["w"]); net.eval()
print(f"Đã nạp {CKPT.parent.name}/{CKPT.name} | epoch {ck['epoch']}")

Đã nạp task8_step2_distance/TCN.pt | epoch 25


In [6]:
#cell 3: nạp landmark gốc của val/test (NaN = mất tay) và tự kiểm tra: acc_all test phải ~93.2% như notebook 13/14
def load_raw(split):
    df = m[m.tap == split].reset_index(drop=True)
    Tm = int(m.khung.max())
    LM = np.full((len(df), Tm, 21, 3), np.nan, np.float32)
    for i, f in enumerate(df.path):
        a = np.load(BASE / f)["lm"].reshape(-1, 21, 3)
        LM[i, :len(a)] = a
    L = torch.tensor(df.khung.values, device=dev)
    return torch.tensor(LM, device=dev), torch.tensor(df.nhan.values, dtype=torch.long, device=dev), L

RAW = {s: load_raw(s) for s in ("val", "test")}
for s, (LM, y, L) in RAW.items():
    print(s, tuple(LM.shape), "| video:", len(y))

LM, y, L = RAW["test"]
with torch.no_grad():
    Xf, has = feats_from_raw(LM)
    pred = net(Xf).argmax(-1)
acc = ((pred == y[:, None]) & has).sum().item() / has.sum().item() * 100
print(f"acc_all test (120 fps, không xoay): {acc:.1f}%  <- mong đợi ~93.2")

val (31, 355, 21, 3) | video: 31
test (33, 355, 21, 3) | video: 33
acc_all test (120 fps, không xoay): 93.2%  <- mong đợi ~93.2


In [7]:
#cell 4: hàm làm mượt (EMA + giữ nhãn + reset khi mất tay) và tự kiểm tra bằng dữ liệu giả
class Smoother:
    """Làm mượt theo luồng. Xử lý n luồng cùng lúc (replay: n = số video; webcam: n = 1) nên chỉ có MỘT bản code."""
    SW_THR = 0.5                                   # lớp mới phải đạt ít nhất 0.5 mới được tính là ứng viên đổi nhãn

    def __init__(self, n, alpha=0.5, margin=0.1, hold=2):
        self.n, self.a, self.mg, self.hold = n, alpha, margin, hold
        self.p = np.zeros((n, 3)); self.init = np.zeros(n, bool)
        self.label = np.full(n, -1); self.cand = np.full(n, -1); self.cnt = np.zeros(n, int)

    def _clear(self, mask):
        self.init[mask] = False; self.label[mask] = -1; self.cand[mask] = -1; self.cnt[mask] = 0

    def update(self, probs, valid):
        """probs: (n,3) xác suất của khung hiện tại; valid: (n,) bool, False = mất tay. Trả về (nhãn (n,), độ tin cậy (n,))."""
        self._clear(~valid)                                              # (3) mất tay -> reset
        new = valid & ~self.init
        self.p[new] = probs[new]; self.init[new] = True                  # khung đầu tiên: lấy luôn, không chờ (để đoán sớm)
        old = valid & ~new
        self.p[old] = self.a * probs[old] + (1 - self.a) * self.p[old]   # (1) EMA
        ar, top = np.arange(self.n), self.p.argmax(1)
        first = valid & (self.label < 0)
        self.label[first] = top[first]                                   # nhãn đầu tiên
        cur = self.label.clip(0)
        diff = valid & (top != self.label)
        strong = diff & (self.p[ar, top] >= self.SW_THR) & (self.p[ar, top] - self.p[ar, cur] >= self.mg)
        self.cnt = np.where(strong & (self.cand == top), self.cnt + 1, np.where(strong, 1, 0))   # (2) đếm số khung liên tiếp
        self.cand = np.where(strong, top, -1)
        sw = strong & (self.cnt >= self.hold)
        self.label[sw], self.cand[sw], self.cnt[sw] = top[sw], -1, 0
        return np.where(valid, self.label, -1), self.p[ar, self.label.clip(0)]

# tự kiểm tra: lớp 1 ổn định, có 1 khung nhiễu (khung 20), mất tay khung 30-32, đổi thật sang lớp 2 từ khung 40
K = 60
P = np.tile([0.1, 0.8, 0.1], (1, K, 1))
P[0, 20] = [0.1, 0.2, 0.7]
P[0, 40:] = [0.1, 0.1, 0.8]
V = np.ones((1, K), bool); V[0, 30:33] = False
sm = Smoother(1, alpha=0.5, margin=0.1, hold=2)
lab = np.array([sm.update(P[:, k], V[:, k])[0][0] for k in range(K)])
raw = P[0].argmax(1)
print("khung nhiễu 20: raw =", raw[20], "| mượt =", lab[20])
print("mất tay 30-32 -> nhãn:", lab[30:33])
print("đổi thật từ khung 40: raw đổi ở khung 40, mượt đổi ở khung", int(np.argmax(lab[35:] == 2)) + 35)
assert lab[20] == 1 and (lab[30:33] == -1).all() and lab[-1] == 2 and lab[34] == 1
print("Cell 4 OK: nhiễu 1 khung bị lọc, mất tay thì reset, đổi thật vẫn theo kịp (trễ ~2 khung webcam).")

khung nhiễu 20: raw = 2 | mượt = 1
mất tay 30-32 -> nhãn: [-1 -1 -1]
đổi thật từ khung 40: raw đổi ở khung 40, mượt đổi ở khung 42
Cell 4 OK: nhiễu 1 khung bị lọc, mất tay thì reset, đổi thật vẫn theo kịp (trễ ~2 khung webcam).


In [8]:
#cell 5: replay val/test ở 30 fps giống webcam (lấy mỗi khung thứ 4, nội suy lên 120 fps, đọc kết quả tại khung webcam thật)
def replay30(split):
    """Trả về xác suất (N,K,3), mask có tay (N,K), nhãn thật (N,) ở K khung webcam 30 fps."""
    LM, y, L = RAW[split]
    step = FPS_DATA // FPS_CAM
    lm30 = LM[:, ::step]                                                    # (N,K,21,3) webcam 30 fps
    A, B = lm30[:, :-1, None], lm30[:, 1:, None]
    j = (torch.arange(1, step + 1, device=dev, dtype=torch.float32) / step).view(1, 1, step, 1, 1)
    mid = A + (B - A) * j                                                   # nội suy tuyến tính, khung cuối của mỗi đoạn = khung webcam mới
    mid = torch.where(torch.isnan(A) & ~torch.isnan(B), B.expand_as(mid), mid)   # tay vừa xuất hiện lại sau khi mất
    N, K = lm30.shape[:2]
    stream = torch.cat([lm30[:, :1], mid.reshape(N, (K - 1) * step, 21, 3)], 1)  # luồng 120 fps như buffer của webcam
    with torch.no_grad():
        X, has = feats_from_raw(stream)
        P = net(X).softmax(-1)[:, ::step]                                   # chỉ đọc tại các khung webcam thật
    idx = torch.arange(P.shape[1], device=dev) * step
    valid = has[:, ::step] & (idx[None] < L[:, None])                       # bỏ phần đệm sau khi video kết thúc
    return P.cpu().numpy(), valid.cpu().numpy(), y.cpu().numpy()

REPLAY = {s: replay30(s) for s in ("val", "test")}
for s, (P, V, y) in REPLAY.items():
    print(s, "| xác suất:", P.shape, "| khung có tay:", int(V.sum()))
P, V, y = REPLAY["test"]
acc30 = ((P.argmax(-1) == y[:, None]) & V).sum() / V.sum() * 100
print(f"acc_all test ở 30 fps (chưa làm mượt): {acc30:.1f}%  <- nên gần 93 (khác chút vì chỉ đọc 1/4 số khung)")

val | xác suất: (31, 89, 3) | khung có tay: 959
test | xác suất: (33, 89, 3) | khung có tay: 1000
acc_all test ở 30 fps (chưa làm mượt): 92.8%  <- nên gần 93 (khác chút vì chỉ đọc 1/4 số khung)


In [9]:
#cell 6: hàm chấm điểm (tiêu chí Decision: ngưỡng 0.5→0.9) + số lần đổi nhãn, và điểm của model nền (không làm mượt)
THRS = [0.5, 0.6, 0.7, 0.8, 0.9]

def run_raw(P, V):
    """Không làm mượt: nhãn = lớp có xác suất cao nhất của từng khung."""
    return np.where(V, P.argmax(-1), -1), P.max(-1)

def run_smooth(P, V, alpha, margin, hold):
    N, K, _ = P.shape
    sm = Smoother(N, alpha, margin, hold)
    lab, conf = np.full((N, K), -1), np.zeros((N, K))
    for k in range(K):                                           # lặp theo thời gian; mọi video chạy song song
        lab[:, k], conf[:, k] = sm.update(P[:, k], V[:, k])
    return lab, conf

def score(lab, conf, V, y):
    ok = (lab == y[:, None]) & V
    t_ms = np.arange(lab.shape[1]) * 1000 / FPS_CAM
    r = {"acc_all": ok.sum() / V.sum() * 100}
    for t in THRS:
        kk = int(round(t * 100))
        hit = ok & (conf >= t)                                   # khung vừa tự tin vừa đúng
        has = hit.any(1)
        r[f"v{kk}"] = has.mean() * 100                           # % video có ít nhất 1 khung như vậy
        r[f"ms{kk}"] = t_ms[hit.argmax(1)[has]].mean() if has.any() else float("nan")   # đoán đúng đầu tiên (ms)
    r["tb"] = float(np.mean([r[f"v{int(round(t * 100))}"] for t in THRS]))
    sel = (conf >= 0.7) & V
    r["accc70"] = (ok & sel).sum() / max(sel.sum(), 1) * 100
    r["cov70"] = sel.sum() / V.sum() * 100
    both = V[:, 1:] & V[:, :-1]
    r["flips"] = ((lab[:, 1:] != lab[:, :-1]) & both).sum(1).mean()   # số lần đổi nhãn trung bình mỗi video
    return r

COLS = ["flips", "acc_all", "tb", "v50", "v60", "accc70", "cov70", "ms60", "ms70"]
BASE_ROWS = {}
for s, (P, V, y) in REPLAY.items():
    BASE_ROWS[s] = score(*run_raw(P, V), V, y)
print("Điểm model nền (không làm mượt), replay 30 fps:")
display(pd.DataFrame(BASE_ROWS).T[COLS].round(1))
print("flips = số lần đổi nhãn trung bình mỗi video (càng thấp càng ít giật); ms = thời điểm đoán đúng đầu tiên.")

Điểm model nền (không làm mượt), replay 30 fps:


,flips,acc_all,tb,v50,v60,accc70,cov70,ms60,ms70
val,0.9,91.4,98.1,100.0,100.0,96.2,90.3,109.7,128.0
test,0.5,92.8,98.2,100.0,100.0,95.5,91.6,107.1,124.2


flips = số lần đổi nhãn trung bình mỗi video (càng thấp càng ít giật); ms = thời điểm đoán đúng đầu tiên.


In [10]:
#cell 7: thử nhiều bộ tham số trên VAL (không đụng test), chọn bộ ít giật nhất mà không hại độ chính xác/độ trễ quá mức
ALPHAS, MARGINS, HOLDS = [0.3, 0.5, 0.7, 1.0], [0.0, 0.1, 0.2], [1, 2, 3]
MAX_ACC_DROP, MAX_DELAY_MS = 0.5, 50      # tiêu chí tham khảo (chỉnh được): acc_all giảm tối đa 0.5 điểm, ms70 tăng tối đa 50 ms (~1.5 khung webcam)
CHOSEN_MANUAL = None                      # muốn tự chọn: ví dụ {"alpha": 0.5, "margin": 0.1, "hold": 2}

Pv, Vv, yv = REPLAY["val"]
base = BASE_ROWS["val"]
rows = []
for a in ALPHAS:
    for mg in MARGINS:
        for h in HOLDS:
            r = score(*run_smooth(Pv, Vv, a, mg, h), Vv, yv)
            rows.append({"alpha": a, "margin": mg, "hold": h, **{k: r[k] for k in COLS},
                         "d_acc": r["acc_all"] - base["acc_all"], "d_ms70": r["ms70"] - base["ms70"],
                         "giam_flips_%": (1 - r["flips"] / max(base["flips"], 1e-9)) * 100})
grid = pd.DataFrame(rows)
ok_rows = grid[(grid.d_acc >= -MAX_ACC_DROP) & (grid.d_ms70 <= MAX_DELAY_MS)]
print(f"Model nền trên val: flips {base['flips']:.2f} | acc_all {base['acc_all']:.1f} | ms70 {base['ms70']:.0f}")
print(f"{len(ok_rows)}/{len(grid)} bộ tham số đạt điều kiện (acc_all giảm ≤ {MAX_ACC_DROP}, ms70 tăng ≤ {MAX_DELAY_MS} ms). Top theo ít giật nhất:")
cols = ["alpha", "margin", "hold", "flips", "giam_flips_%", "acc_all", "d_acc", "ms70", "d_ms70", "tb"]
display(ok_rows.sort_values(["flips", "ms70"]).head(8)[cols].round(2))

if CHOSEN_MANUAL is not None:
    CHOSEN = CHOSEN_MANUAL
elif len(ok_rows):
    b = ok_rows.sort_values(["flips", "ms70"]).iloc[0]
    CHOSEN = {"alpha": float(b.alpha), "margin": float(b.margin), "hold": int(b.hold)}
else:
    CHOSEN = {"alpha": 0.5, "margin": 0.1, "hold": 2}
    print("CẢNH BÁO: không bộ nào đạt điều kiện; tạm dùng bộ mặc định đề xuất ban đầu. Xem lại bảng grid hoặc chỉnh MAX_*.")
print("Bộ tham số được chọn:", CHOSEN)

Model nền trên val: flips 0.90 | acc_all 91.4 | ms70 128
0/36 bộ tham số đạt điều kiện (acc_all giảm ≤ 0.5, ms70 tăng ≤ 50 ms). Top theo ít giật nhất:


,alpha,margin,hold,flips,giam_flips_%,acc_all,d_acc,ms70,d_ms70,tb


CẢNH BÁO: không bộ nào đạt điều kiện; tạm dùng bộ mặc định đề xuất ban đầu. Xem lại bảng grid hoặc chỉnh MAX_*.
Bộ tham số được chọn: {'alpha': 0.5, 'margin': 0.1, 'hold': 2}


In [11]:
#cell 8: báo cáo trên TEST: model nền (không làm mượt) so với bộ tham số đã chọn ở cell 7
Pt, Vt, yt = REPLAY["test"]
res_t = {"không làm mượt": BASE_ROWS["test"], "có làm mượt": score(*run_smooth(Pt, Vt, **CHOSEN), Vt, yt)}
display(pd.DataFrame(res_t).T[COLS].round(1))

a, b = res_t["không làm mượt"], res_t["có làm mượt"]
cut = (1 - b["flips"] / max(a["flips"], 1e-9)) * 100
print(f"Số lần đổi nhãn mỗi video: {a['flips']:.2f} -> {b['flips']:.2f} (giảm {cut:.0f}%)")
print(f"Δ acc_all {b['acc_all'] - a['acc_all']:+.1f} | Δ TB 5 ngưỡng {b['tb'] - a['tb']:+.1f} | Δ acc khi tin cậy ≥0.7 {b['accc70'] - a['accc70']:+.1f} | Δ độ phủ 0.7 {b['cov70'] - a['cov70']:+.1f}")
print(f"Độ trễ: đoán đúng đầu tiên ở ngưỡng 0.6 {b['ms60'] - a['ms60']:+.0f} ms | ở 0.7 {b['ms70'] - a['ms70']:+.0f} ms")
if b["flips"] < a["flips"] and b["acc_all"] - a["acc_all"] >= -MAX_ACC_DROP and b["ms70"] - a["ms70"] <= MAX_DELAY_MS:
    print("KẾT QUẢ SỐ LIỆU: ĐẠT tiêu chí sơ bộ (ít giật hơn, độ chính xác và độ trễ trong mức cho phép). Cần xác nhận bằng webcam (cell 11).")
else:
    print("KẾT QUẢ SỐ LIỆU: CHƯA ĐẠT tiêu chí sơ bộ; xem lại tham số ở cell 7 hoặc giữ bản không làm mượt.")
print("Lưu ý: test chỉ 33 video, chênh dưới ~1 điểm chưa chắc chắn; tiêu chí chỉ để tham khảo, quyết định cuối cùng sau khi thử webcam.")

,flips,acc_all,tb,v50,v60,accc70,cov70,ms60,ms70
không làm mượt,0.5,92.8,98.2,100.0,100.0,95.5,91.6,107.1,124.2
có làm mượt,0.3,90.9,98.2,100.0,100.0,96.5,89.3,127.3,138.4


Số lần đổi nhãn mỗi video: 0.52 -> 0.27 (giảm 47%)
Δ acc_all -1.9 | Δ TB 5 ngưỡng +0.0 | Δ acc khi tin cậy ≥0.7 +1.0 | Δ độ phủ 0.7 -2.3
Độ trễ: đoán đúng đầu tiên ở ngưỡng 0.6 +20 ms | ở 0.7 +14 ms
KẾT QUẢ SỐ LIỆU: CHƯA ĐẠT tiêu chí sơ bộ; xem lại tham số ở cell 7 hoặc giữ bản không làm mượt.
Lưu ý: test chỉ 33 video, chênh dưới ~1 điểm chưa chắc chắn; tiêu chí chỉ để tham khảo, quyết định cuối cùng sau khi thử webcam.


In [12]:
#cell 9: MediaPipe (CPU), buffer nội suy 120 fps và hàm dự đoán xác suất cho khung webcam hiện tại (giống 08_WebcamWide / 13)
import cv2
import mediapipe as mp

landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()     # mốc thời gian dùng chung: timestamp của MediaPipe phải luôn tăng, kể cả khi chạy lại cell 11

def detect(frame_bgr, ts_ms):
    """Trả về tọa độ ảnh chuẩn hóa (21,3), hoặc None nếu mất tay."""
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not res.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in res.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)          # mỗi phần tử (21,3); NaN = mất tay
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    """Thêm một khung webcam (lm hoặc None, t tính bằng giây) vào buf, chèn các khung nội suy 120 fps."""
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

@torch.no_grad()
def predict_probs():
    """Xác suất 3 lớp (numpy (3,)) của khung cuối, hoặc None nếu khung cuối không có tay."""
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    X, _ = feats_from_raw(torch.from_numpy(lm)[None].to(dev))
    return net(X).softmax(-1)[0, -1].cpu().numpy()

with torch.no_grad(): net(torch.zeros(1, 128, 546, device=dev))   # khởi động CUDA để khung đầu không bị giật
print("Đã tạo HandLandmarker (CPU); sẵn sàng nội suy", FPS_MODEL, "fps | cửa sổ", WIN, "khung")

Đã tạo HandLandmarker (CPU); sẵn sàng nội suy 120 fps | cửa sổ 128 khung


In [13]:
#cell 10: giao diện (giống 08_WebcamWide): màu theo lớp, chữ ĐEN trên nền màu, khung tay + khung xương, thêm dòng trạng thái làm mượt
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}   # BGR: paper = đỏ, rock = xanh dương, scissors = xanh lá
GRAY, YELLOW, BLACK = (190, 190, 190), (150, 255, 255), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]

def put_label(img, text, x, y, bg, scale=0.8):
    """Chữ đen trên hộp màu; (x, y) là góc dưới-trái của hộp."""
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y - h - base - 6), (x + w + 10, y + 2), bg, -1)
    cv2.putText(img, text, (x + 5, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, img_lm, color, label, mirror=True):
    """Vẽ khung xương, khung bao quanh tay và nhãn (cùng màu) lên ảnh."""
    h, w = img.shape[:2]
    px, py = (img_lm[:, 0] * w).astype(int), (img_lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 3, cv2.LINE_AA)
    for i in range(21):
        cv2.circle(img, (px[i], py[i]), 6, color, -1, cv2.LINE_AA)
        cv2.circle(img, (px[i], py[i]), 6, BLACK, 1, cv2.LINE_AA)
    x0, y0, x1, y1 = px.min() - 15, py.min() - 15, px.max() + 15, py.max() + 15
    cv2.rectangle(img, (x0, y0), (x1, y1), color, 3, cv2.LINE_AA)
    ly = y1 + 34 if (x0 < 330 and y0 < 165) else y0      # tránh đè lên khối thông tin ở góc trên trái
    put_label(img, label, max(x0, 0), min(max(ly, 32), h - 5), color)

def draw_hud(img, fps, thr, smooth_on, ch_raw, ch_sm):
    """FPS, ngưỡng, trạng thái làm mượt, bộ đếm đổi nhãn và chú thích màu của 3 lớp."""
    put_label(img, f"FPS {fps:.0f}", 10, 35, YELLOW)
    put_label(img, f"Nguong >= {thr:.2f}", 10, 75, YELLOW)
    put_label(img, f"Lam muot: {'BAT' if smooth_on else 'TAT'} (s)", 10, 115, YELLOW)
    put_label(img, f"Doi nhan raw/muot: {ch_raw}/{ch_sm}", 10, 155, YELLOW)
    for i, name in enumerate(CLASSES):
        put_label(img, name, img.shape[1] - 125, img.shape[0] - 10 - 40 * (2 - i), COLORS[i])
print("Đã định nghĩa giao diện.")

Đã định nghĩa giao diện.


In [14]:
#cell 11: chạy webcam — q: thoát | s: bật/tắt làm mượt | [ ]: giảm/tăng ngưỡng hiển thị màu (mỗi lần 0.05)
CAM_ID, MIRROR, THR = 0, True, 0.60                    # ngưỡng hiển thị màu 0.6 (08_WebcamWide là 0.70)
PARAMS = CHOSEN                                         # bộ tham số chọn ở cell 7; muốn thử bộ khác thì gán tay ở đây
smooth_on = True
sm = Smoother(1, **PARAMS)
print("Tham số làm mượt:", PARAMS)

cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))     # MJPG giúp webcam đạt 30 fps ở 640x480
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
fps, last = 0.0, time.perf_counter()
ch_raw = ch_sm = 0; last_raw = last_sm = -1             # đếm số lần đổi nhãn kể từ lúc tay xuất hiện
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        img_lm = detect(frame, int((now - T0) * 1000))   # MediaPipe nhận ảnh gốc (không lật), giống lúc trích landmark
        push(img_lm, now)
        probs = predict_probs() if img_lm is not None else None
        valid = probs is not None
        lab_s, conf_s = sm.update((probs if valid else np.zeros(3))[None], np.array([valid]))   # luôn cập nhật để bật/tắt không bị lệch
        if valid:
            raw_c, raw_conf, sm_c, sm_conf = int(probs.argmax()), float(probs.max()), int(lab_s[0]), float(conf_s[0])
            ch_raw += int(last_raw >= 0 and raw_c != last_raw); ch_sm += int(last_sm >= 0 and sm_c != last_sm)
            last_raw, last_sm = raw_c, sm_c
        else:
            ch_raw = ch_sm = 0; last_raw = last_sm = -1  # mất tay -> đếm lại từ đầu

        disp = cv2.flip(frame, 1) if MIRROR else frame   # chỉ lật để hiển thị cho dễ chơi
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        if not valid:
            put_label(disp, "no hand", 10, 195, GRAY)
        else:
            c, conf = (sm_c, sm_conf) if smooth_on else (raw_c, raw_conf)
            sure = conf >= THR
            draw_hand(disp, img_lm, COLORS[c] if sure else GRAY,
                      f"{CLASSES[c]} {conf:.0%}" if sure else f"{CLASSES[c]}? {conf:.0%}", MIRROR)
        draw_hud(disp, fps, THR, smooth_on, ch_raw, ch_sm)

        cv2.imshow("RPS webcam - Step 4 lam muot (q: thoat | s: lam muot)", disp)
        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == ord("s"): smooth_on = not smooth_on
        if k == ord("["): THR = max(0.05, THR - 0.05)
        if k == ord("]"): THR = min(0.95, THR + 0.05)
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam.")

Tham số làm mượt: {'alpha': 0.5, 'margin': 0.1, 'hold': 2}
Đã đóng webcam.


## Cách quyết định giữ hay bỏ Step 4

- Ưu tiên **giảm số lần đổi nhãn** (cell 8, và bộ đếm raw/mượt trên webcam) mà **không giảm độ chính xác** và **không tăng độ trễ quá mức** (đoán đúng đầu tiên, ms).
- Trên webcam: giữ yên một cử chỉ vài giây ở nhiều góc, nhấn `s` để so hai chế độ. Sau đó thử **ra tay thật nhanh** (rock → paper, v.v.) để xem nhãn mượt có theo kịp không. Game cần đoán sớm nên đây là phép thử quan trọng nhất.
- Nếu thấy nhãn đổi chậm quá thì tăng `alpha` hoặc giảm `hold`/`margin`; nếu còn giật thì làm ngược lại. Gán tay ở cell 11 (`PARAMS`) để thử nhanh, không cần chạy lại cell 7.
- Ghi vào `Decision.docx`: tham số đã chọn, số lần đổi nhãn trước/sau, Δ độ chính xác, Δ ms, nhận xét webcam. Nếu giữ thì đánh dấu Bước 4 là [Chốt].
- Lưu ý: tham số chỉ chọn trên val (31 video) và đánh giá trên test (33 video), chênh nhỏ chưa chắc chắn; tiêu chí chỉ mang tính tham khảo.